In [1]:
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / ".projectroot").exists())

# V-Dem Governance : EDA + Preparation

- Raw: `../Datasets/training_data/raw/vdem/V-Dem-CY-Full+Others-v16.csv`
- Output: `../Datasets/training_data/clean/vdem/vdem_clean.csv`
- I keep a few genuinely distinct governance indices and reshape to one row per country-year keyed on ISO3.

## 1. Setup and Load

I import the libraries and set the raw and output paths.

In [2]:
import os
import numpy as np
import pandas as pd

RAW = str(ROOT / "data/raw/vdem/V-Dem-CY-Full+Others-v16.csv")
OUT_DIR = str(ROOT / "data/interim/vdem")
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 170)

I set one consistent, high-resolution style for every chart (resolution, fonts, sizes, colors).

In [3]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 130, 'savefig.dpi': 200,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 14, 'axes.titleweight': 'bold', 'axes.labelsize': 12,
    'axes.edgecolor': '#444444', 'axes.linewidth': 0.8,
    'axes.grid': True, 'grid.color': '#E9E9E9', 'grid.linewidth': 0.8,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
})
NAVY, BLUE, RED = '#1F3864', '#2E75B6', '#C0392B'

V-Dem builds its indices from thousands of expert ratings run through a Bayesian measurement model. Most of these indices are heavily redundant. Only 5 genuinely distinct axes are kept: liberal democracy (the overall 0 to 1 score), regime type, corruption (higher is worse), physical safety, and gender equality.

In [4]:
SELECTED = {
    'v2x_libdem': 'lib_democracy',
    'v2x_regime': 'regime_type',
    'v2x_corr':   'corruption',
    'v2x_clphy':  'physical_safety',
    'v2x_gender': 'gender_equality',
}
print(len(SELECTED), 'indices selected')

5 indices selected


I load only the columns I need from the file, the country and year keys plus the 5 selected indices.

In [5]:
vd = pd.read_csv(RAW, usecols=['country_text_id', 'country_name', 'year'] + list(SELECTED))
print('loaded ->', vd.shape, '| year range:', vd['year'].min(), '-', vd['year'].max())
vd.head(3)

loaded -> (28092, 8) | year range: 1789 - 2025


,country_name,country_text_id,year,v2x_libdem,v2x_regime,v2x_clphy,v2x_corr,v2x_gender
0,Mexico,MEX,1789,0.044,NaN,0.314,0.68,0.121
1,Mexico,MEX,1790,0.044,NaN,0.314,0.68,0.121
2,Mexico,MEX,1791,0.044,NaN,0.314,0.68,0.121


`vd` = V-Dem country-year table, one row per country per year.

- Each row = one country in one year: the country name, its ISO3 code (country_text_id), the year, and the 5 governance indices.
- Columns: country_text_id is the ISO3 key; lib_democracy (0 to 1, higher = more democratic), regime_type (0 to 3 ordinal, higher = more democratic), corruption (0 to 1, higher = worse), physical_safety (0 to 1, higher = safer), gender_equality (0 to 1, higher = better).
- Example: the first row, Mexico in 1789, scores lib_democracy 0.044 (very low), regime_type missing, corruption 0.68 (high), physical_safety 0.314, and gender_equality 0.121.
- The file spans 28,092 country-years from 1789 to 2025 (early rows like this Mexico 1789 row are historical, with some indices NaN that far back), I use 2015 onward, where coverage is complete.

## 2. Data Discovery

I filter to 2015 onward to confirm  that the 5 indices are well covered, and that they are distinct (no duplicates).

In [6]:
v15 = vd[vd['year'] >= 2015].copy()
feat = list(SELECTED)
print('2015+ rows:', len(v15), '| countries:', v15['country_text_id'].nunique(), '| years:', v15['year'].min(), '-', v15['year'].max())
print('coverage %:', (v15[feat].notna().mean() * 100).round(1).to_dict())
s5 = v15[feat].corr(method='spearman').round(2)
s5.index = [SELECTED[i] for i in s5.index]; s5.columns = [SELECTED[c] for c in s5.columns]
s5

2015+ rows: 1969 | countries: 179 | years: 2015 - 2025
coverage %: {'v2x_libdem': 100.0, 'v2x_regime': 100.0, 'v2x_corr': 100.0, 'v2x_clphy': 100.0, 'v2x_gender': 99.9}


,lib_democracy,regime_type,corruption,physical_safety,gender_equality
lib_democracy,1.00,0.91,-0.75,0.89,0.85
regime_type,0.91,1.00,-0.64,0.79,0.81
corruption,-0.75,-0.64,1.00,-0.77,-0.69
physical_safety,0.89,0.79,-0.77,1.00,0.81
gender_equality,0.85,0.81,-0.69,0.81,1.00


Coverage is essentially 100% across all 5 (gender at 99.9%) over 1,969 country-years and 179 countries, so nothing is dropped. 

## 3. Data Preparation

I rename the 5 indices to clean names, keep the keys plus features, and sort by country and year.

In [7]:
vdem = v15.rename(columns={**SELECTED, 'country_text_id': 'iso3', 'country_name': 'country'})
vdem = vdem[['iso3', 'country', 'year'] + list(SELECTED.values())].sort_values(['iso3', 'year']).reset_index(drop=True)
print('vdem ->', vdem.shape, '| countries:', vdem.iso3.nunique(), '| years:', vdem.year.min(), '-', vdem.year.max())
vdem[vdem.iso3.isin(['USA', 'RUS', 'UKR']) & vdem.year.isin([2015, 2023])].sort_values(['iso3', 'year'])

vdem -> (1969, 8) | countries: 179 | years: 2015 - 2025


,iso3,country,year,lib_democracy,regime_type,corruption,physical_safety,gender_equality
1463,RUS,Russia,2015,0.109,1.0,0.712,0.368,0.686
1471,RUS,Russia,2023,0.065,1.0,0.780,0.258,0.572
1826,UKR,Ukraine,2015,0.228,1.0,0.784,0.454,0.814
1834,UKR,Ukraine,2023,0.239,1.0,0.610,0.619,0.889
1848,USA,United States of America,2015,0.848,3.0,0.053,0.954,0.898
1856,USA,United States of America,2023,0.789,3.0,0.055,0.949,0.897


The clean table is 1,969 country-years across 179 countries, 2015 to 2025, with 8 columns (iso3, country, year plus the 5 features). The sample confirms every direction reads correctly, the USA scores high on lib_democracy (0.79 to 0.85), regime_type 3 (liberal democracy), near-zero corruption and high physical_safety. Russia is the mirror image, low lib_democracy (near 0.1) declining, regime_type 1 (electoral autocracy), high and rising corruption (0.71 to 0.78). The features seem to behave exactly as expected.

## 4. Exploratory Data Analysis

A world map of the liberal democracy index (0 to 1).

In [8]:
import geopandas as gpd
from ipywidgets import interact, Dropdown, Layout
WSTYLE = {'description_width': '110px'}; WIDE = Layout(width='430px')
world = gpd.read_file(str(ROOT / "data/external/geodata/ne_110m_admin_0_countries.geojson"))[['ISO_A3_EH', 'geometry']]
year_opts = ['Mean (all years)'] + [str(y) for y in sorted(vdem.year.unique())]

def democracy_map(year='Mean (all years)'):
    if year == 'Mean (all years)':
        d = vdem.groupby('iso3')['lib_democracy'].mean().reset_index()
        title = 'Mean liberal democracy index by country, 2015 to 2025'
    else:
        d = vdem[vdem.year == int(year)][['iso3', 'lib_democracy']]
        title = f'Liberal democracy index by country, {year}'
    m = world.merge(d, left_on='ISO_A3_EH', right_on='iso3', how='left')
    fig, ax = plt.subplots(figsize=(13, 6.5))
    m.plot(column='lib_democracy', ax=ax, cmap='RdYlBu', legend=True, vmin=0, vmax=1,
           edgecolor='#999999', linewidth=0.3, missing_kwds={'color': '#EEEEEE', 'label': 'no data'})
    ax.grid(False); ax.set_title(title, pad=12); ax.axis('off')
    fig.tight_layout(); plt.show()

interact(democracy_map, year=Dropdown(options=year_opts, value='Mean (all years)', description='Year', style=WSTYLE, layout=WIDE))

interactive(children=(Dropdown(description='Year', layout=Layout(width='430px'), options=('Mean (all years)', …

<function __main__.democracy_map(year='Mean (all years)')>

I load the GPR target, join it to the V-Dem features on the labeled countries, and rank the 5 features by Spearman correlation with GPR.

In [9]:
gpr = pd.read_csv(str(ROOT / "data/interim/gpr/gpr_monthly.csv"))
gpr['year'] = pd.to_datetime(gpr['month']).dt.year
target = gpr[(gpr.year >= 2015) & (gpr.year <= 2023)].groupby(['iso3', 'year'])['gpr'].mean().reset_index()

ms = vdem.merge(target, on=['iso3', 'year'], how='inner')
print('labeled country-years:', len(ms), '| countries:', ms.iso3.nunique())
feats5 = list(SELECTED.values())
ms[feats5 + ['gpr']].corr(method='spearman')['gpr'].drop('gpr').sort_values(ascending=False).to_frame('spearman_with_gpr').round(3)

labeled country-years: 396 | countries: 44


,spearman_with_gpr
regime_type,0.099
gender_equality,0.012
physical_safety,-0.039
lib_democracy,-0.063
corruption,-0.080


Across 396 labeled country-years (44 countries) all 5 governance features correlate essentially zero with GPR, which is natural, since GPR is a news and military driven index, so there will be little direct linear relationship, so I chose to keep all 5 features.

## 5. Validate and Save

I check the final country-year table for duplicate iso3-year keys and any missing values before saving. I keep iso3 and year as the keys plus the 5 features (I drop the country name, since the master join keys on ISO3).

In [10]:
out = vdem[['iso3', 'year'] + list(SELECTED.values())].sort_values(['iso3', 'year']).reset_index(drop=True)
print('rows:', len(out), '| countries:', out.iso3.nunique(), '| years:', out.year.min(), '-', out.year.max())
print('duplicate iso3-year keys:', int(out.duplicated(['iso3', 'year']).sum()))
out.isna().sum().to_frame('n_missing')

rows: 1969 | countries: 179 | years: 2015 - 2025
duplicate iso3-year keys: 0


,n_missing
iso3,0
year,0
lib_democracy,0
regime_type,0
corruption,0
physical_safety,0
gender_equality,1


The table is clean: 1,969 country-years across 179 countries, 2015 to 2025, with no duplicate iso3-year keys. The only missing value is a single gender_equality cell (the 99.9% coverage seen earlier), I leave it as NaN for the model to handle. 

I confirm the keys are unique and save the clean per-country-year table.

In [11]:
assert out.duplicated(['iso3', 'year']).sum() == 0, 'duplicate iso3-year keys!'
os.makedirs(OUT_DIR, exist_ok=True)
path = f'{OUT_DIR}/vdem_clean.csv'
out.to_csv(path, index=False)
print('saved:', path, '|', out.shape, '| countries:', out.iso3.nunique())
out.head()

saved: /Users/oussamaennaciri/Documents/Education/Academique/Regis/MSDS692 S40 Data Science Practicum/data/interim/vdem/vdem_clean.csv | (1969, 7) | countries: 179


,iso3,year,lib_democracy,regime_type,corruption,physical_safety,gender_equality
0,AFG,2015,0.201,1.0,0.904,0.406,0.483
1,AFG,2016,0.188,1.0,0.906,0.403,0.472
2,AFG,2017,0.183,1.0,0.898,0.419,0.488
3,AFG,2018,0.176,1.0,0.896,0.429,0.500
4,AFG,2019,0.162,1.0,0.898,0.395,0.489


Claude (Anthropic) was used only as a collaborator for writing code and polishing the notebooks. All analytical decisions, interpretations, and research were conducted independently by me.